In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated , List # Annotated adds extra metadata
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages

# Add memory
from langgraph.checkpoint.memory import MemorySaver # This stores state in RAM

In [3]:

class ChatState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

In [4]:
llm = ChatOpenAI()

In [6]:
def chat_node(state: ChatState):
    # take user query from state
    messages = state['messages']
    
    # send to llm
    response = llm.invoke(messages)
    
    # response store in state
    return {"messages":[response]}

In [7]:
# Create Checkpoint using MemorySaver
checkpoint = MemorySaver()

# define graph
graph = StateGraph(ChatState)

# ADD NODE
graph.add_node("chat_node", chat_node)

# add edges
graph.add_edge(START, 'chat_node')
graph.add_edge("chat_node", END)

# compile
workflow = graph.compile(checkpointer=checkpoint)

In [ ]:
initial_state = {"messages": [HumanMessage(content="What is capital of india")]}
workflow.invoke(initial_state)

{'messages': [HumanMessage(content='What is capital of india', additional_kwargs={}, response_metadata={}, id='fdedc296-9b99-4701-8bff-a307fe481471'),
  AIMessage(content='The capital of India is New Delhi.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 12, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DHqtkA12OrPfvnKt0c29ZGssNlmHi', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cd7d0-71a0-7781-8149-e0f10a03ebb8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 8, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_de

In [ ]:
thread_id="1"

while True:
    user_query = input('type Here')
    print(f"user: {user_query}")
    
    if user_query.strip().lower() in ['exit', 'quite', 'bye']:
        break
        
    config = {'configurable':{"thread_id": thread_id}}
    response = workflow.invoke({'messages':[HumanMessage(user_query)]}, config=config)
    print("AI: ",response["messages"][-1].content)

user: Hello my name is pratik
AI:  Hello Pratik, nice to meet you! How can I assist you today?
user: bye


In [10]:
list(workflow.get_state_history(config=config))


[StateSnapshot(values={'messages': [HumanMessage(content='Hello my name is pratik', additional_kwargs={}, response_metadata={}, id='a9836605-2db6-4dee-8c24-6ceb65d6f8ec'), AIMessage(content='Hello Pratik, nice to meet you! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 13, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DHsHq1z9hKK6fqHEQdpTOn6VRb7bh', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cd821-e8cf-7251-9f4a-1d11ec19bc8e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 17, 'total_tokens': 30, 'input_token_details': {